# HW01-C — Airflow Scheduled Pipeline

A pipeline that only runs when you remember to click it is a chore.

Here you turn the SQL work into an Airflow DAG. The DAG refreshes your materialized view, validates it, and writes a run report.

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

- Airflow DAGs: https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/dags.html
- Airflow Variables: https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/variables.html
- Airflow best practices: https://airflow.apache.org/docs/apache-airflow/stable/best-practices.html

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

In [1]:
from pathlib import Path
import os, re, textwrap

PROJECT = Path.cwd()
if PROJECT.name != '03-benavaz' and (PROJECT / '03-benavaz').exists():
    PROJECT = PROJECT / '03-benavaz'
os.chdir(PROJECT)

for path in ['dags', 'reports', 'screenshots']:
    (PROJECT / path).mkdir(exist_ok=True)

student_id = os.getenv('QBC12_STUDENT_ID', 'amirhossein_sa').strip()
safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', student_id.lower())
DAG_ID = f'qbc12_hw01_{safe_student}_airbnb_pipeline'
STUDENT_SCHEMA = f'student_{safe_student}'
DAG_ID, STUDENT_SCHEMA


('qbc12_hw01_amirhossein_sa_airbnb_pipeline', 'student_amirhossein_sa')

## 1. DAG design

Build this chain:

```text
read_config → refresh_summary → validate_summary → branch → success/failure report
```

Database credentials must come from Airflow Variables.

In [ ]:
# 1.1 Create dags/<DAG_ID>.py with imports, DAG metadata, make_engine(), and read_config.
# The DAG source below also includes the later refresh, validation, branch, and report tasks
# so running the notebook top-to-bottom always leaves a complete Airflow DAG.


## 2. Refresh task

The refresh task should recreate your materialized view in Postgres. Do not move the full dataset into Python.

In [5]:
# 2.1 Confirm refresh_summary(config) is implemented in the DAG file.
# It recreates the materialized view and indexes inside Postgres, then returns a small dict.

dag_text = (PROJECT / 'dags' / f'{DAG_ID}.py').read_text(encoding='utf-8')
assert 'def refresh_summary(config' in dag_text
assert 'create materialized view' in dag_text.lower()
assert 'create unique index' in dag_text.lower()
assert 'create index' in dag_text.lower()
print('refresh_summary is implemented.')


refresh_summary is implemented.


## 3. Validation task

Required checks:

- row_count > 0
- null_neighbourhoods == 0
- bad_prices == 0
- bad_availability == 0

In [6]:
# 3.1 Confirm validate_summary(config) returns the required checks plus passed=True/False.

dag_text = (PROJECT / 'dags' / f'{DAG_ID}.py').read_text(encoding='utf-8')
for check_name in ['row_count', 'null_neighbourhoods', 'bad_prices', 'bad_availability', 'passed']:
    assert check_name in dag_text, f'Missing validation field: {check_name}'
print('validate_summary checks are implemented.')


validate_summary checks are implemented.


## 4. Branching and reports

Success and failure should not look the same.

Use `@task.branch` to choose the report path.

In [7]:
# 4.1 Confirm branching and success/failure report tasks are implemented.
# The failure path writes its report and then raises ValueError so Airflow marks the run failed.

dag_text = (PROJECT / 'dags' / f'{DAG_ID}.py').read_text(encoding='utf-8')
for snippet in ['@task.branch', 'def choose_report_path', 'def write_success_report', 'def write_failure_report', 'raise ValueError']:
    assert snippet in dag_text, f'Missing DAG snippet: {snippet}'
print('Branching and report tasks are implemented.')


Branching and report tasks are implemented.


In [8]:
# Syntax check. This is not a full Airflow run.
import py_compile

dag_path = PROJECT / 'dags' / f'{DAG_ID}.py'
assert dag_path.exists(), f'Missing DAG file: {dag_path}'
py_compile.compile(str(dag_path), doraise=True)
print('DAG compiles:', dag_path)

DAG compiles: /home/ahs/Personal Projects/mlops-bootcamp/03-benavaz/dags/qbc12_hw01_amirhossein_sa_airbnb_pipeline.py


## 5. Shared Airflow run

In shared Airflow:

1. find your DAG
2. unpause it
3. trigger it manually
4. inspect Graph view
5. inspect logs
6. confirm the materialized view was refreshed

Screenshots:

```text
screenshots/airflow_dag_graph.png
screenshots/airflow_success_run.png
```

In [9]:
# 5.1 Write reports/hw01_c_airflow.md.
# Fill QBC12_AIRFLOW_URL and QBC12_AIRFLOW_SUCCESS_TS in your environment after the shared run
# if you want this cell to stamp the final server evidence automatically.

from datetime import datetime, timezone

report_path = PROJECT / 'reports' / 'hw01_c_airflow.md'
airflow_url = os.getenv('QBC12_AIRFLOW_URL', 'shared QBC12 Airflow URL from the homework page')
success_ts = os.getenv('QBC12_AIRFLOW_SUCCESS_TS', 'pending shared Airflow trigger')
validation_result = os.getenv(
    'QBC12_AIRFLOW_VALIDATION_RESULT',
    'the DAG validates row_count > 0, null_neighbourhoods == 0, bad_prices == 0, and bad_availability == 0',
)
report = f'''# HW01-C Airflow Scheduled Pipeline

- DAG id: `{DAG_ID}`
- Airflow URL: {airflow_url}
- Successful run timestamp: {success_ts}
- Refreshed object name: `{STUDENT_SCHEMA}.mv_airbnb_neighbourhood_summary`
- Validation result: {validation_result}
- Screenshot paths:
  - `screenshots/airflow_dag_graph.png`
  - `screenshots/airflow_success_run.png`

## Notes

The DAG file was generated from this notebook and syntax-checked locally. After triggering it in shared Airflow, update the timestamp above and place the required screenshots at the listed paths.

Report generated locally at: {datetime.now(timezone.utc).isoformat()}
'''
report_path.write_text(report, encoding='utf-8')
print(f'Wrote report: {report_path}')


Wrote report: /home/ahs/Personal Projects/mlops-bootcamp/03-benavaz/reports/hw01_c_airflow.md


In [ ]:
for file in [f'dags/{DAG_ID}.py', 'reports/hw01_c_airflow.md']:
    assert Path(file).exists(), f'Missing {file}'
print('Basic deliverables exist.')

## Commit

```bash
git add dags reports screenshots notebooks
git commit -m "HW01-C Airflow scheduled pipeline"
```